In [2]:
%pip install sentence-transformers pymupdf python-docx streamlit

In [3]:
import pandas as pd
import numpy as np
import sklearn
import sentence_transformers
import fitz
import docx
import streamlit as st

print("All libraries imported successfully!")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Sentence Transformers imported successfully!")
print("PDF and DOCX libraries imported successfully!")
print("Streamlit imported successfully!")

All libraries imported successfully!
Pandas version: 2.3.3
NumPy version: 2.2.6
Scikit-learn version: 1.7.2
Sentence Transformers imported successfully!
PDF and DOCX libraries imported successfully!
Streamlit imported successfully!


In [4]:
import pandas as pd

resumes = [
    {
        "resume_id": 1,
        "name": "Candidate 1",
        "resume_text": """
        Python developer with knowledge of Python, SQL, MySQL,
        Pandas, NumPy, Power BI and Excel.
        Completed projects in data analysis and visualization.
        Familiar with data cleaning, reporting and machine learning basics.
        """
    },
    {
        "resume_id": 2,
        "name": "Candidate 2",
        "resume_text": """
        Java developer with skills in Java, HTML, CSS, JavaScript,
        SQL and Spring Boot.
        Developed web applications and REST APIs.
        Interested in software development and backend programming.
        """
    },
    {
        "resume_id": 3,
        "name": "Candidate 3",
        "resume_text": """
        Data analyst skilled in Excel, SQL, Python, Power BI,
        Tableau, statistics and dashboard development.
        Experienced in data visualization, reporting and data cleaning.
        """
    }
]

resume_df = pd.DataFrame(resumes)

resume_df

,resume_id,name,resume_text
0,1,Candidate 1,\n Python developer with knowledge of P...
1,2,Candidate 2,"\n Java developer with skills in Java, ..."
2,3,Candidate 3,"\n Data analyst skilled in Excel, SQL, ..."


In [5]:
job_description = """
Job Title: Data Analyst

We are looking for a Data Analyst with knowledge of
Python, SQL, Excel, Power BI and data visualization.

Responsibilities:
- Analyze and clean datasets.
- Create dashboards and reports.
- Use SQL to extract and manage data.
- Perform exploratory data analysis.
- Communicate insights using charts and visualizations.

Required Skills:
Python, SQL, Excel, Power BI, Pandas, NumPy,
statistics and data visualization.
"""

print(job_description)


Job Title: Data Analyst

We are looking for a Data Analyst with knowledge of
Python, SQL, Excel, Power BI and data visualization.

Responsibilities:
- Analyze and clean datasets.
- Create dashboards and reports.
- Use SQL to extract and manage data.
- Perform exploratory data analysis.
- Communicate insights using charts and visualizations.

Required Skills:
Python, SQL, Excel, Power BI, Pandas, NumPy,
statistics and data visualization.



In [6]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9+#.\s]", "", text)
    return text.strip()

# Clean all resume texts
resume_df["cleaned_resume"] = resume_df["resume_text"].apply(clean_text)

# Clean the job description
cleaned_job_description = clean_text(job_description)

print("Cleaned Resume Text:")
print(resume_df["cleaned_resume"][0])

print("\nCleaned Job Description:")
print(cleaned_job_description)


Cleaned Resume Text:
python developer with knowledge of python sql mysql pandas numpy power bi and excel. completed projects in data analysis and visualization. familiar with data cleaning reporting and machine learning basics.

Cleaned Job Description:
job title data analyst we are looking for a data analyst with knowledge of python sql excel power bi and data visualization. responsibilities  analyze and clean datasets.  create dashboards and reports.  use sql to extract and manage data.  perform exploratory data analysis.  communicate insights using charts and visualizations. required skills python sql excel power bi pandas numpy statistics and data visualization.


In [7]:

import re

skill_keywords = [
    # Data and programming
    "python", "java", "c++", "c#", "javascript", "sql", "mysql",
    "pandas", "numpy", "excel", "power bi", "tableau", "statistics",
    "machine learning", "deep learning", "data analysis",
    "data visualization", "data cleaning", "reporting",

    # Web and software
    "html", "css", "react", "node.js", "spring boot",
    "rest api", "git", "github", "software development",
    "testing", "debugging",

    # Cloud and infrastructure
    "aws", "azure", "google cloud", "docker", "kubernetes",
    "linux", "networking", "cybersecurity",

    # Business and office
    "communication", "leadership", "project management",
    "problem solving", "customer service", "sales",
    "marketing", "digital marketing", "seo", "content writing",
    "recruitment", "human resources", "payroll",
    "accounting", "financial analysis", "budgeting",
    "inventory management", "supply chain", "operations",
    "mechanical design", "autocad", "quality control"
]

def extract_skills(text):
    text = text.lower()
    found_skills = []

    for skill in skill_keywords:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
        if re.search(pattern, text):
            found_skills.append(skill)

    return sorted(set(found_skills))

resume_df["extracted_skills"] = resume_df["cleaned_resume"].apply(extract_skills)

print("Resume Skills:")
for _, row in resume_df.iterrows():
    print(f"\n{row['name']}: {row['extracted_skills']}")

print("\nJob Description Skills:")
print(extract_skills(cleaned_job_description))


Resume Skills:

Candidate 1: ['data analysis', 'data cleaning', 'excel', 'machine learning', 'mysql', 'numpy', 'pandas', 'power bi', 'python', 'reporting', 'sql']

Candidate 2: ['css', 'html', 'java', 'javascript', 'software development', 'spring boot', 'sql']

Candidate 3: ['data cleaning', 'data visualization', 'excel', 'power bi', 'python', 'reporting', 'sql', 'statistics', 'tableau']

Job Description Skills:
['data analysis', 'data visualization', 'excel', 'numpy', 'pandas', 'power bi', 'python', 'sql', 'statistics']


In [8]:

job_skills = set(extract_skills(cleaned_job_description))

def compare_skills(resume_text):
    resume_skills = set(extract_skills(resume_text))

    matched = sorted(resume_skills.intersection(job_skills))
    missing = sorted(job_skills - resume_skills)

    if len(job_skills) > 0:
        skill_match_percentage = round(
            len(matched) / len(job_skills) * 100, 2
        )
    else:
        skill_match_percentage = 0.0

    return pd.Series({
        "matched_skills": matched,
        "missing_skills": missing,
        "skill_match_percentage": skill_match_percentage
    })

skill_results = resume_df["cleaned_resume"].apply(compare_skills)

resume_df = pd.concat([resume_df, skill_results], axis=1)

display(
    resume_df[
        ["name", "matched_skills", "missing_skills",
         "skill_match_percentage"]
    ]
)


,name,matched_skills,missing_skills,skill_match_percentage
0,Candidate 1,"[data analysis, excel, numpy, pandas, power bi...","[data visualization, statistics]",77.78
1,Candidate 2,[sql],"[data analysis, data visualization, excel, num...",11.11
2,Candidate 3,"[data visualization, excel, power bi, python, ...","[data analysis, numpy, pandas]",66.67


In [9]:

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Sentence Transformer model loaded successfully!")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\dhuma\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dhuma\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer model loaded successfully!


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Convert resumes into numerical embeddings
resume_embeddings = model.encode(
    resume_df["cleaned_resume"].tolist(),
    convert_to_numpy=True
)

# Convert job description into an embedding
job_embedding = model.encode(
    [cleaned_job_description],
    convert_to_numpy=True
)

# Calculate semantic similarity
similarity_scores = cosine_similarity(
    resume_embeddings,
    job_embedding
).flatten()

# Store scores as percentages
resume_df["semantic_similarity"] = (
    similarity_scores * 100
).round(2)

display(
    resume_df[
        ["name", "skill_match_percentage", "semantic_similarity"]
    ]
)

,name,skill_match_percentage,semantic_similarity
0,Candidate 1,77.78,79.580002
1,Candidate 2,11.11,28.469999
2,Candidate 3,66.67,87.120003


In [11]:
# Combine skill matching and semantic similarity
resume_df["final_match_score"] = (
    0.5 * resume_df["skill_match_percentage"]
    + 0.5 * resume_df["semantic_similarity"]
).round(2)

# Sort resumes by final match score
ranked_resumes = resume_df.sort_values(
    by="final_match_score",
    ascending=False
)

display(
    ranked_resumes[
        [
            "name",
            "skill_match_percentage",
            "semantic_similarity",
            "final_match_score"
        ]
    ]
)

,name,skill_match_percentage,semantic_similarity,final_match_score
0,Candidate 1,77.78,79.580002,78.68
2,Candidate 3,66.67,87.120003,76.90
1,Candidate 2,11.11,28.469999,19.79


In [12]:
# Find the best-matching candidate
best_candidate = ranked_resumes.iloc[0]

print("Best Matching Candidate:", best_candidate["name"])
print("Final Match Score:", best_candidate["final_match_score"], "%")

print("\nMatched Skills:")
print(best_candidate["matched_skills"])

print("\nMissing Skills:")
print(best_candidate["missing_skills"])

Best Matching Candidate: Candidate 1
Final Match Score: 78.68 %

Matched Skills:
['data analysis', 'excel', 'numpy', 'pandas', 'power bi', 'python', 'sql']

Missing Skills:
['data visualization', 'statistics']


In [13]:
import pymupdf

def extract_text_from_pdf(file_path):
    extracted_text = ""

    with pymupdf.open(file_path) as pdf:
        for page in pdf:
            extracted_text += page.get_text() + "\n"

    return extracted_text.strip()

In [14]:
pdf_path = r"C:\Users\dhuma\Desktop\Data Analytics\Vaishnavi Dhumal DA Resume..pdf"

pdf_text = extract_text_from_pdf(pdf_path)

print(pdf_text[:2000])

VAISHNAVI DHUMAL   
 Pune, India | dhumalvaishnavi003@gmail.com | +91-9284173533 |  
 LinkedIn: https://www.linkedin.com/in/vaishnavi-dhumal26  
 GitHub: https://github.com/Vaishnavidhumal-26  
SUMMARY 
Motivated and detail-oriented Computer-Engineering student with good knowledge of Data    
analytics. Excel, SQL, Power BI, Python, Statistics and Tableau. Skilled in data cleaning, 
visualization, dashboard creation, and generating business insights from data. Learner with 
problem-solving abilities and work in academic and self-learning projects. Seeking an 
opportunity to contribute analytical and technical skills in a Data Analytics role.  
   
TECHNICAL SKILLS   
• 
Data Analytics Tools: Microsoft Excel, Power BI, SQL, MYSQL, Tableau, Statistics.  
• 
Power BI Skills: Data Modelling, DAX, Power Query, Dashboard Development, 
Data Visualization  
 
•  
Programming Languages:  Python  
 
 PROJECTS   
  1. Sales Analytics Dashboard | Power BI 
• 
Built an interactive dashboard to anal

In [15]:
# Clean the extracted resume text
actual_resume_cleaned = clean_text(pdf_text)

# Extract skills from the actual resume
actual_resume_skills = extract_skills(actual_resume_cleaned)

print("Extracted Skills:")
print(actual_resume_skills)

print("\nTotal Skills Found:", len(actual_resume_skills))

Extracted Skills:
['customer service', 'data analysis', 'data cleaning', 'data visualization', 'excel', 'github', 'mysql', 'numpy', 'pandas', 'power bi', 'python', 'sales', 'sql', 'statistics', 'tableau']

Total Skills Found: 15


In [16]:
# Extract skills required by the job description
job_skills = set(extract_skills(cleaned_job_description))

# Compare actual resume skills with job skills
matched_skills = sorted(
    set(actual_resume_skills).intersection(job_skills)
)

missing_skills = sorted(
    job_skills - set(actual_resume_skills)
)

# Calculate skill match percentage
if job_skills:
    actual_skill_match = round(
        len(matched_skills) / len(job_skills) * 100, 2
    )
else:
    actual_skill_match = 0.0

print("Job Skills:", sorted(job_skills))
print("\nMatched Skills:", matched_skills)
print("\nMissing Skills:", missing_skills)
print("\nSkill Match Percentage:", actual_skill_match, "%")

Job Skills: ['data analysis', 'data visualization', 'excel', 'numpy', 'pandas', 'power bi', 'python', 'sql', 'statistics']

Matched Skills: ['data analysis', 'data visualization', 'excel', 'numpy', 'pandas', 'power bi', 'python', 'sql', 'statistics']

Missing Skills: []

Skill Match Percentage: 100.0 %


In [17]:
from sklearn.metrics.pairwise import cosine_similarity

# Create embeddings for the actual resume and job description
actual_resume_embedding = model.encode(
    [actual_resume_cleaned],
    convert_to_numpy=True
)

job_embedding = model.encode(
    [cleaned_job_description],
    convert_to_numpy=True
)

# Calculate semantic similarity
actual_similarity = cosine_similarity(
    actual_resume_embedding,
    job_embedding
)[0][0]

actual_semantic_score = round(actual_similarity * 100, 2)

print("Skill Match Percentage:", actual_skill_match, "%")
print("Semantic Similarity:", actual_semantic_score, "%")

Skill Match Percentage: 100.0 %
Semantic Similarity: 74.61 %


In [18]:
# Calculate the combined resume match score
actual_final_score = round(
    0.5 * actual_skill_match
    + 0.5 * actual_semantic_score,
    2
)

print("Resume Analysis Report")
print("----------------------")
print("Skill Match:", actual_skill_match, "%")
print("Semantic Similarity:", actual_semantic_score, "%")
print("Final Match Score:", actual_final_score, "%")

Resume Analysis Report
----------------------
Skill Match: 100.0 %
Semantic Similarity: 74.61 %
Final Match Score: 87.3 %


In [19]:
from docx import Document

def extract_text_from_docx(file_path):
    document = Document(file_path)

    paragraphs = [
        paragraph.text
        for paragraph in document.paragraphs
        if paragraph.text.strip()
    ]

    # Also extract text from tables
    for table in document.tables:
        for row in table.rows:
            for cell in row.cells:
                if cell.text.strip():
                    paragraphs.append(cell.text)

    return "\n".join(paragraphs).strip()

In [20]:
docx_path = r"C:\Users\dhuma\Desktop\Data Analytics\Vaishnavi Dhumal AI Resume.docx"

docx_text = extract_text_from_docx(docx_path)

print(docx_text[:2000])

VAISHNAVI DHUMAL
Computer Science & Engineering Graduate | Python | AI/ML | Data Analytics
 Pune, India | dhumalvaishnavi003@gmail.com | +91-9284173533 |  
 LinkedIn: https://www.linkedin.com/in/vaishnavi-dhumal26  
 GitHub: https://github.com/Vaishnavidhumal-26  
PROFESSIONAL SUMMARY
Computer Science and Engineering graduate with good knowledge in Python, data     preprocessing, exploratory data analysis, statistical analysis and data visualization. Familiar with Machine Learning concepts and Generative AI fundamentals, with a strong interest in developing AI-based applications. Experienced in working with real-world datasets using Pandas, NumPy, Matplotlib, Seaborn and SciPy. Strong analytical and problem-solving skills with a willingness to learn LLMs, Prompt Engineering, RAG and modern AI frameworks. Seeking an AI Developer Intern opportunity to apply Python and AI/ML knowledge to practical projects.
TECHNICAL SKILLS
Programming: Python, SQL, MySQL
AI & Machine Learning: Machine Le

In [21]:
def extract_text_from_txt(file_path):
    encodings = ["utf-8", "latin-1"]

    for encoding in encodings:
        try:
            with open(
                file_path,
                "r",
                encoding=encoding
            ) as file:
                return file.read().strip()

        except UnicodeDecodeError:
            continue

    raise ValueError("Unable to read this TXT file.")

In [23]:
import os

def extract_resume_text(file_path):
    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":
        return extract_text_from_pdf(file_path)

    elif extension == ".docx":
        return extract_text_from_docx(file_path)

    else:
        raise ValueError(
            "Unsupported file format. Upload PDF or DOCX."
        )

In [24]:
resume_path = r"C:\Users\dhuma\Downloads\Vaishnavi Dhumal Data Analytics Resume (1).pdf"

resume_text = extract_resume_text(resume_path)

print(resume_text[:1000])

VAISHNAVI DHUMAL   
 Pune, India | dhumalvaishnavi003@gmail.com | +91-9284173533 |  
 LinkedIn: https://www.linkedin.com/in/vaishnavi-dhumal26  
 GitHub: https://github.com/Vaishnavidhumal-26  
SUMMARY 
Motivated and detail-oriented Computer-Engineering student with good knowledge of Data    
analytics. Excel, SQL, Power BI, Python, Statistics and Tableau. Skilled in data cleaning, 
visualization, dashboard creation, and generating business insights from data. Learner with 
problem-solving abilities and work in academic and self-learning projects. Seeking an 
opportunity to contribute analytical and technical skills in a Data Analytics role.  
   
TECHNICAL SKILLS   
• 
Data Analytics Tools: Microsoft Excel, Power BI, SQL, MYSQL, Tableau, Statistics.  
• 
Power BI Skills: Data Modelling, DAX, Power Query, Dashboard Development, 
Data Visualization  
 
•  
Programming Languages:  Python  
 
 PROJECTS   
  1. Sales Analytics Dashboard | Power BI 
• 
Built an interactive dashboard to anal

In [25]:
# Clean the extracted resume text
resume_cleaned = clean_text(resume_text)

# Extract skills from the resume
resume_skills = extract_skills(resume_cleaned)

# Compare resume skills with the current job description
matched_skills = sorted(
    set(resume_skills).intersection(job_skills)
)

missing_skills = sorted(
    job_skills - set(resume_skills)
)

# Calculate skill match percentage
skill_match = round(
    len(matched_skills) / len(job_skills) * 100, 2
) if job_skills else 0.0

print("Resume Skills:", resume_skills)
print("\nMatched Skills:", matched_skills)
print("\nMissing Skills:", missing_skills)
print("\nSkill Match Percentage:", skill_match, "%")

Resume Skills: ['customer service', 'data analysis', 'data cleaning', 'data visualization', 'excel', 'github', 'mysql', 'numpy', 'pandas', 'power bi', 'python', 'sales', 'sql', 'statistics', 'tableau']

Matched Skills: ['data analysis', 'data visualization', 'excel', 'numpy', 'pandas', 'power bi', 'python', 'sql', 'statistics']

Missing Skills: []

Skill Match Percentage: 100.0 %


In [27]:
# Enter any job description
new_job_description = input(
    "Paste the Job Description here: "
)

# Clean the job description
new_job_cleaned = clean_text(new_job_description)

# Extract required skills
new_job_skills = set(
    extract_skills(new_job_cleaned)
)

# Compare with the resume skills
matched_skills = sorted(
    set(resume_skills).intersection(new_job_skills)
)

missing_skills = sorted(
    new_job_skills - set(resume_skills)
)

# Calculate skill match percentage
if new_job_skills:
    new_skill_match = round(
        len(matched_skills) / len(new_job_skills) * 100, 2
    )
else:
    new_skill_match = 0.0

print("\n--- Resume Matching Report ---")
print("Required Skills:", sorted(new_job_skills))
print("Matched Skills:", matched_skills)
print("Missing Skills:", missing_skills)
print("Skill Match:", new_skill_match, "%")

Paste the Job Description here:  Paste the Job Description here:  Job Title: AI Developer Intern  Required Skills: Python, Machine Learning, Generative AI, SQL, Pandas, NumPy, Prompt Engineering, LLMs.  Responsibilities: Build AI applications, preprocess data, evaluate machine learning models, and work with NLP.  --- Resume Matching Report --- Required Skills: ['machine learning', 'numpy', 'pandas', 'python', 'sql'] Matched Skills: ['numpy', 'pandas', 'python', 'sql'] Missing Skills: ['machine learning'] Skill Match: 80.0 %



--- Resume Matching Report ---
Required Skills: ['machine learning', 'numpy', 'pandas', 'python', 'sql']
Matched Skills: ['numpy', 'pandas', 'python', 'sql']
Missing Skills: ['machine learning']
Skill Match: 80.0 %


In [28]:
additional_skills = [
    "machine learning",
    "generative ai",
    "prompt engineering",
    "llm",
    "llms",
    "natural language processing",
    "nlp",
    "deep learning",
    "neural networks",
    "model evaluation",
    "data preprocessing",
    "artificial intelligence",
    "rag",
    "transformers",
    "pytorch",
    "tensorflow",
    "scikit-learn"
]

# Add new skills without duplicates
skill_keywords = sorted(
    set(skill_keywords + additional_skills)
)

print("Updated skill keywords:", len(skill_keywords))
print(additional_skills)

Updated skill keywords: 75
['machine learning', 'generative ai', 'prompt engineering', 'llm', 'llms', 'natural language processing', 'nlp', 'deep learning', 'neural networks', 'model evaluation', 'data preprocessing', 'artificial intelligence', 'rag', 'transformers', 'pytorch', 'tensorflow', 'scikit-learn']


In [29]:
# Re-extract skills using the updated keyword list
resume_skills = extract_skills(resume_cleaned)

# Extract skills from the AI Developer job description
new_job_skills = set(
    extract_skills(new_job_cleaned)
)

# Compare resume skills with job requirements
matched_skills = sorted(
    set(resume_skills).intersection(new_job_skills)
)

missing_skills = sorted(
    new_job_skills - set(resume_skills)
)

# Calculate skill match
if new_job_skills:
    new_skill_match = round(
        len(matched_skills) / len(new_job_skills) * 100,
        2
    )
else:
    new_skill_match = 0.0

print("Resume Skills:", resume_skills)
print("\nRequired Skills:", sorted(new_job_skills))
print("\nMatched Skills:", matched_skills)
print("\nMissing Skills:", missing_skills)
print("\nSkill Match:", new_skill_match, "%")


Resume Skills: ['customer service', 'data analysis', 'data cleaning', 'data visualization', 'excel', 'github', 'mysql', 'numpy', 'pandas', 'power bi', 'python', 'sales', 'sql', 'statistics', 'tableau']

Required Skills: ['generative ai', 'llms', 'machine learning', 'nlp', 'numpy', 'pandas', 'prompt engineering', 'python', 'sql']

Matched Skills: ['numpy', 'pandas', 'python', 'sql']

Missing Skills: ['generative ai', 'llms', 'machine learning', 'nlp', 'prompt engineering']

Skill Match: 44.44 %


In [30]:
# Create embeddings for the current resume and new job description
resume_embedding = model.encode(
    [resume_cleaned],
    convert_to_numpy=True
)

new_job_embedding = model.encode(
    [new_job_cleaned],
    convert_to_numpy=True
)

# Calculate semantic similarity
semantic_score = cosine_similarity(
    resume_embedding,
    new_job_embedding
)[0][0]

semantic_percentage = round(
    semantic_score * 100, 2
)

# Calculate combined score
final_score = round(
    0.5 * new_skill_match
    + 0.5 * semantic_percentage,
    2
)

print("AI Developer Resume Report")
print("--------------------------")
print("Skill Match:", new_skill_match, "%")
print("Semantic Similarity:", semantic_percentage, "%")
print("Combined Match Score:", final_score, "%")

AI Developer Resume Report
--------------------------
Skill Match: 44.44 %
Semantic Similarity: 50.7 %
Combined Match Score: 47.57 %


In [31]:
def match_resume_to_job(resume_text, job_description):
    # Clean both texts
    clean_resume = clean_text(resume_text)
    clean_job = clean_text(job_description)

    # Extract skills
    resume_skills = set(extract_skills(clean_resume))
    required_skills = set(extract_skills(clean_job))

    # Compare skills
    matched = sorted(resume_skills & required_skills)
    missing = sorted(required_skills - resume_skills)

    # Calculate skill match
    if required_skills:
        skill_score = round(
            len(matched) / len(required_skills) * 100, 2
        )
    else:
        skill_score = 0.0

    # Calculate semantic similarity
    resume_embedding = model.encode(
        [clean_resume], convert_to_numpy=True
    )
    job_embedding = model.encode(
        [clean_job], convert_to_numpy=True
    )

    semantic_score = round(
        float(cosine_similarity(
            resume_embedding, job_embedding
        )[0][0]) * 100, 2
    )

    # Experimental combined score
    combined_score = round(
        0.5 * skill_score + 0.5 * semantic_score, 2
    )

    return {
        "skill_match_percentage": skill_score,
        "semantic_similarity": semantic_score,
        "combined_match_score": combined_score,
        "matched_skills": matched,
        "missing_skills": missing
    }

print("Resume matching function created successfully!")

Resume matching function created successfully!


In [32]:
# Match your actual resume with the AI Developer job description
result = match_resume_to_job(
    resume_text,
    new_job_description
)

print("RESUME MATCHING REPORT")
print("----------------------")
print("Skill Match:", result["skill_match_percentage"], "%")
print("Semantic Similarity:", result["semantic_similarity"], "%")
print("Combined Match Score:", result["combined_match_score"], "%")
print("\nMatched Skills:", result["matched_skills"])
print("Missing Skills:", result["missing_skills"])

RESUME MATCHING REPORT
----------------------
Skill Match: 44.44 %
Semantic Similarity: 50.7 %
Combined Match Score: 47.57 %

Matched Skills: ['numpy', 'pandas', 'python', 'sql']
Missing Skills: ['generative ai', 'llms', 'machine learning', 'nlp', 'prompt engineering']


In [33]:

import os
print(os.getcwd())


C:\Users\dhuma\Desktop\Data Science\Deep Learning\Resume_Job_Matching
